# Boosting Methods for Round Winner Prediction

This notebook implements and compares three boosting algorithms:
1. **AdaBoost** (Adaptive Boosting)
2. **Gradient Boosting**
3. **XGBoost** (Extreme Gradient Boosting)

## Table of Contents
1. [Introduction](#introduction)
2. [Data Loading and Preparation](#data-loading)
3. [AdaBoost Implementation](#adaboost)
4. [Gradient Boosting Implementation](#gradient-boosting)
5. [XGBoost Implementation](#xgboost)
6. [Model Comparison](#comparison)
7. [Visualizations](#visualizations)
8. [Conclusions](#conclusions)


## Introduction

Boosting is an ensemble learning technique that combines multiple weak learners (typically shallow decision trees) into a strong learner. The key idea is to sequentially train models, where each new model focuses on correcting the errors made by previous models.

### Key Differences:
- **AdaBoost**: Adjusts sample weights based on misclassifications
- **Gradient Boosting**: Fits new models to the residual errors
- **XGBoost**: Optimized gradient boosting with regularization and advanced features


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

print("All libraries imported successfully!")


## Data Loading and Preparation


In [ ]:
# Load all match data
data_dir = '../bayes_dt/hltv_data'

# Get all rounds and deaths files
rounds_files = glob.glob(os.path.join(data_dir, '*_rounds.csv'))
deaths_files = glob.glob(os.path.join(data_dir, '*_deaths.csv'))

print(f"Found {len(rounds_files)} match files")

# Load and combine all rounds data
rounds_dfs = []
for file in rounds_files:
    df = pd.read_csv(file)
    match_name = os.path.basename(file).replace('_rounds.csv', '')
    df['match_id'] = match_name
    rounds_dfs.append(df)

rounds_df = pd.concat(rounds_dfs, ignore_index=True)
print(f"Total rounds loaded: {len(rounds_df)}")

# Load and combine all deaths data
deaths_dfs = []
for file in deaths_files:
    df = pd.read_csv(file)
    match_name = os.path.basename(file).replace('_deaths.csv', '')
    df['match_id'] = match_name
    deaths_dfs.append(df)

deaths_df = pd.concat(deaths_dfs, ignore_index=True)
print(f"Total deaths loaded: {len(deaths_df)}")


In [ ]:
# Feature engineering function
def create_round_features_fixed(rounds_df, deaths_df):
    """
    Create features for round winner prediction from rounds and deaths data.
    """
    features_list = []
    
    for idx, round_row in rounds_df.iterrows():
        match_id = round_row['match_id']
        round_num = round_row['round_num']
        
        # Get deaths for this round
        round_deaths = deaths_df[
            (deaths_df['match_id'] == match_id)
        ].copy()
        
        if len(round_deaths) == 0:
            continue
            
        # Calculate features
        features = {
            'match_id': match_id,
            'round_num': round_num,
            'winning_team': round_row['winning_team'],
            
            # Bomb features
            'bomb_planted': int(round_row['bomb_planted']),
            
            # Kill features
            'total_kills': len(round_deaths),
            'headshot_rate': round_deaths['headshot'].mean() if len(round_deaths) > 0 else 0,
            
            # Damage features
            'avg_damage': round_deaths['dmg_health'].mean(),
            'total_damage': round_deaths['dmg_health'].sum(),
            
            # Distance features
            'avg_distance': round_deaths['distance'].mean(),
            
            # Weapon-specific kills
            'ak47_kills': (round_deaths['weapon'] == 'ak47').sum(),
            'awp_kills': (round_deaths['weapon'] == 'awp').sum(),
            'm4a1_kills': (round_deaths['weapon'].isin(['m4a1', 'm4a1_silencer'])).sum(),
            
            # Special kill types
            'headshot_kills': round_deaths['headshot'].sum(),
            'noscope_kills': round_deaths['noscope'].sum(),
            'thrusmoke_kills': round_deaths['thrusmoke'].sum(),
        }
        
        features_list.append(features)
    
    return pd.DataFrame(features_list)

# Create features
print("Creating features...")
features_df = create_round_features_fixed(rounds_df, deaths_df)
features_df_clean = features_df.dropna(subset=['winning_team'])

print(f"Features created for {len(features_df_clean)} rounds")
print(f"\nFeature columns: {features_df_clean.columns.tolist()}")


In [ ]:
# Display sample data
print("Sample of feature data:")
print(features_df_clean.head(10))

# Check target distribution
print("\nTarget variable distribution:")
print(features_df_clean['winning_team'].value_counts())


In [ ]:
# Prepare data for modeling
numeric_features = [
    'bomb_planted', 'total_kills', 'headshot_rate', 
    'avg_damage', 'total_damage', 'avg_distance', 
    'ak47_kills', 'awp_kills', 'm4a1_kills',
    'headshot_kills', 'noscope_kills', 'thrusmoke_kills'
]

X = features_df_clean[numeric_features].fillna(0)
y = features_df_clean['winning_team']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")
print(f"\nTraining set target distribution:")
print(y_train.value_counts())
print(f"\nTesting set target distribution:")
print(y_test.value_counts())


## AdaBoost Implementation

AdaBoost (Adaptive Boosting) works by:
1. Training a weak learner on the data
2. Increasing weights of misclassified samples
3. Training the next weak learner on the reweighted data
4. Combining all weak learners with weighted voting


In [ ]:
# Test different AdaBoost configurations
adaboost_results = []

# Parameter grid
n_estimators_list = [50, 100, 200]
learning_rates = [0.5, 1.0, 1.5]

print("Training AdaBoost models with different parameters...\n")

for n_est in n_estimators_list:
    for lr in learning_rates:
        # Create and train model
        ada_model = AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1),
            n_estimators=n_est,
            learning_rate=lr,
            random_state=42,
            algorithm='SAMME'
        )
        
        ada_model.fit(X_train, y_train)
        y_pred = ada_model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        
        adaboost_results.append({
            'n_estimators': n_est,
            'learning_rate': lr,
            'accuracy': accuracy,
            'model': ada_model
        })
        
        print(f"n_estimators={n_est}, learning_rate={lr}: Accuracy = {accuracy:.4f}")

# Get best model
best_ada = max(adaboost_results, key=lambda x: x['accuracy'])
print(f"\nBest AdaBoost: n_estimators={best_ada['n_estimators']}, lr={best_ada['learning_rate']}, accuracy={best_ada['accuracy']:.4f}")


In [ ]:
# Train final AdaBoost model with best parameters
ada_best_model = best_ada['model']
ada_predictions = ada_best_model.predict(X_test)
ada_accuracy = accuracy_score(y_test, ada_predictions)

print("AdaBoost Classification Report:")
print(classification_report(y_test, ada_predictions))

print("\nAdaBoost Confusion Matrix:")
ada_cm = confusion_matrix(y_test, ada_predictions)
print(ada_cm)


## Gradient Boosting Implementation

Gradient Boosting works by:
1. Training a model on the data
2. Computing residual errors
3. Training the next model to predict the residuals
4. Combining models additively


In [ ]:
# Test different Gradient Boosting configurations
gb_results = []

# Parameter grid
n_estimators_list = [50, 100, 200]
learning_rates_gb = [0.05, 0.1, 0.2]
max_depths = [3, 5]

print("Training Gradient Boosting models with different parameters...\n")

for n_est in n_estimators_list:
    for lr in learning_rates_gb:
        for depth in max_depths:
            # Create and train model
            gb_model = GradientBoostingClassifier(
                n_estimators=n_est,
                learning_rate=lr,
                max_depth=depth,
                random_state=42
            )
            
            gb_model.fit(X_train, y_train)
            y_pred = gb_model.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            
            gb_results.append({
                'n_estimators': n_est,
                'learning_rate': lr,
                'max_depth': depth,
                'accuracy': accuracy,
                'model': gb_model
            })
            
            print(f"n_estimators={n_est}, lr={lr}, depth={depth}: Accuracy = {accuracy:.4f}")

# Get best model
best_gb = max(gb_results, key=lambda x: x['accuracy'])
print(f"\nBest Gradient Boosting: n_estimators={best_gb['n_estimators']}, lr={best_gb['learning_rate']}, depth={best_gb['max_depth']}, accuracy={best_gb['accuracy']:.4f}")


In [ ]:
# Train final Gradient Boosting model with best parameters
gb_best_model = best_gb['model']
gb_predictions = gb_best_model.predict(X_test)
gb_accuracy = accuracy_score(y_test, gb_predictions)

print("Gradient Boosting Classification Report:")
print(classification_report(y_test, gb_predictions))

print("\nGradient Boosting Confusion Matrix:")
gb_cm = confusion_matrix(y_test, gb_predictions)
print(gb_cm)


## XGBoost Implementation

XGBoost (Extreme Gradient Boosting) is an optimized gradient boosting implementation with:
- Regularization to prevent overfitting
- Parallel processing
- Tree pruning
- Built-in cross-validation


In [ ]:
# Encode target variable for XGBoost
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

print(f"Classes: {le.classes_}")


In [ ]:
# Test different XGBoost configurations
xgb_results = []

# Parameter grid
n_estimators_list = [50, 100, 200]
learning_rates_xgb = [0.05, 0.1, 0.2]
max_depths_xgb = [3, 5, 7]

print("Training XGBoost models with different parameters...\n")

for n_est in n_estimators_list:
    for lr in learning_rates_xgb:
        for depth in max_depths_xgb:
            # Create and train model
            xgb_model = xgb.XGBClassifier(
                n_estimators=n_est,
                learning_rate=lr,
                max_depth=depth,
                random_state=42,
                eval_metric='logloss'
            )
            
            xgb_model.fit(X_train, y_train_encoded)
            y_pred = xgb_model.predict(X_test)
            accuracy = accuracy_score(y_test_encoded, y_pred)
            
            xgb_results.append({
                'n_estimators': n_est,
                'learning_rate': lr,
                'max_depth': depth,
                'accuracy': accuracy,
                'model': xgb_model
            })
            
            print(f"n_estimators={n_est}, lr={lr}, depth={depth}: Accuracy = {accuracy:.4f}")

# Get best model
best_xgb = max(xgb_results, key=lambda x: x['accuracy'])
print(f"\nBest XGBoost: n_estimators={best_xgb['n_estimators']}, lr={best_xgb['learning_rate']}, depth={best_xgb['max_depth']}, accuracy={best_xgb['accuracy']:.4f}")


In [ ]:
# Train final XGBoost model with best parameters
xgb_best_model = best_xgb['model']
xgb_predictions = xgb_best_model.predict(X_test)
xgb_accuracy = accuracy_score(y_test_encoded, xgb_predictions)

print("XGBoost Classification Report:")
print(classification_report(y_test_encoded, xgb_predictions, target_names=le.classes_))

print("\nXGBoost Confusion Matrix:")
xgb_cm = confusion_matrix(y_test_encoded, xgb_predictions)
print(xgb_cm)


## Model Comparison


In [ ]:
# Compare all three models
comparison_df = pd.DataFrame({
    'Model': ['AdaBoost', 'Gradient Boosting', 'XGBoost'],
    'Accuracy': [ada_accuracy, gb_accuracy, xgb_accuracy],
    'Best Parameters': [
        f"n_est={best_ada['n_estimators']}, lr={best_ada['learning_rate']}",
        f"n_est={best_gb['n_estimators']}, lr={best_gb['learning_rate']}, depth={best_gb['max_depth']}",
        f"n_est={best_xgb['n_estimators']}, lr={best_xgb['learning_rate']}, depth={best_xgb['max_depth']}"
    ]
})

print("Model Comparison:")
print(comparison_df.to_string(index=False))

# Determine best overall model
best_model_name = comparison_df.loc[comparison_df['Accuracy'].idxmax(), 'Model']
best_accuracy = comparison_df['Accuracy'].max()
print(f"\n{'='*60}")
print(f"Best Overall Model: {best_model_name} with accuracy {best_accuracy:.4f}")
print(f"{'='*60}")


## Visualizations


In [ ]:
# Save directory for images
img_dir = '../../src/views/img'
os.makedirs(img_dir, exist_ok=True)

print(f"Images will be saved to: {img_dir}")


In [ ]:
# 1. Boosting Overview Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Weak learner vs Strong learner
ax1 = axes[0]
weak_acc = [0.52, 0.53, 0.51, 0.54, 0.52]
strong_acc = [0.65]
x_weak = np.arange(len(weak_acc))

ax1.bar(x_weak, weak_acc, alpha=0.6, label='Weak Learners', color='lightcoral')
ax1.axhline(y=strong_acc[0], color='darkgreen', linewidth=3, label='Strong Learner (Ensemble)', linestyle='--')
ax1.set_xlabel('Weak Learner Index', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title('Weak Learners vs Strong Learner', fontsize=14, fontweight='bold')
ax1.legend()
ax1.set_ylim([0.4, 0.7])
ax1.grid(True, alpha=0.3)

# Iterative improvement
ax2 = axes[1]
iterations = np.arange(1, 11)
training_error = 0.5 * np.exp(-0.15 * iterations) + 0.05

ax2.plot(iterations, training_error, marker='o', linewidth=2, markersize=8, color='darkblue')
ax2.set_xlabel('Number of Estimators', fontsize=12)
ax2.set_ylabel('Training Error', fontsize=12)
ax2.set_title('Boosting: Iterative Error Reduction', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(img_dir, 'boosting_overview.png'), dpi=300, bbox_inches='tight')
plt.close()

print("✓ Saved boosting_overview.png")


In [ ]:
# 2. Boosting Concept Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# AdaBoost
ax1 = axes[0]
rounds = ['Round 1', 'Round 2', 'Round 3', 'Final']
weights = [1.0, 1.5, 2.0, 1.0]
colors = ['lightblue', 'orange', 'lightcoral', 'darkgreen']
ax1.bar(rounds, weights, color=colors, alpha=0.7)
ax1.set_ylabel('Sample Weight', fontsize=12)
ax1.set_title('AdaBoost:\nSample Reweighting', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# Gradient Boosting
ax2 = axes[1]
stages = ['Model 1', 'Model 2', 'Model 3', 'Model 4']
residuals = [0.45, 0.30, 0.18, 0.10]
ax2.plot(stages, residuals, marker='o', linewidth=2, markersize=10, color='darkblue')
ax2.set_ylabel('Residual Error', fontsize=12)
ax2.set_title('Gradient Boosting:\nResidual Minimization', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

# XGBoost
ax3 = axes[2]
features_xgb = ['Regularization', 'Parallel\nProcessing', 'Tree\nPruning', 'Built-in\nCV']
importance = [0.9, 0.95, 0.85, 0.8]
ax3.barh(features_xgb, importance, color='purple', alpha=0.7)
ax3.set_xlabel('Importance', fontsize=12)
ax3.set_title('XGBoost:\nKey Features', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(os.path.join(img_dir, 'boosting_concept.png'), dpi=300, bbox_inches='tight')
plt.close()

print("✓ Saved boosting_concept.png")


In [ ]:
# 3. Data Sample Visualization
fig, ax = plt.subplots(figsize=(14, 6))

# Display first 10 rows of feature data
sample_data = features_df_clean[['winning_team', 'bomb_planted', 'total_kills', 'headshot_rate', 
                                   'avg_damage', 'ak47_kills', 'awp_kills']].head(10)

# Create table
table = ax.table(cellText=sample_data.values,
                colLabels=sample_data.columns,
                cellLoc='center',
                loc='center',
                bbox=[0, 0, 1, 1])

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

# Style header
for i in range(len(sample_data.columns)):
    table[(0, i)].set_facecolor('#4CAF50')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Alternate row colors
for i in range(1, len(sample_data) + 1):
    for j in range(len(sample_data.columns)):
        if i % 2 == 0:
            table[(i, j)].set_facecolor('#f0f0f0')
        else:
            table[(i, j)].set_facecolor('white')

ax.axis('off')
ax.set_title('Sample CS2 Round Data for Boosting Models', fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(os.path.join(img_dir, 'boosting_data_sample.png'), dpi=300, bbox_inches='tight')
plt.close()

print("✓ Saved boosting_data_sample.png")


In [ ]:
# 4. Train/Test Split Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Training set distribution
ax1 = axes[0]
train_counts = y_train.value_counts()
ax1.bar(train_counts.index, train_counts.values, color=['#FF6B6B', '#4ECDC4'], alpha=0.8)
ax1.set_xlabel('Team', fontsize=12)
ax1.set_ylabel('Number of Rounds', fontsize=12)
ax1.set_title(f'Training Set Distribution\n(n={len(y_train)})', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(train_counts.values):
    ax1.text(i, v + 5, str(v), ha='center', fontweight='bold')

# Testing set distribution
ax2 = axes[1]
test_counts = y_test.value_counts()
ax2.bar(test_counts.index, test_counts.values, color=['#FF6B6B', '#4ECDC4'], alpha=0.8)
ax2.set_xlabel('Team', fontsize=12)
ax2.set_ylabel('Number of Rounds', fontsize=12)
ax2.set_title(f'Testing Set Distribution\n(n={len(y_test)})', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(test_counts.values):
    ax2.text(i, v + 2, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(img_dir, 'boosting_train_test_split.png'), dpi=300, bbox_inches='tight')
plt.close()

print("✓ Saved boosting_train_test_split.png")


In [ ]:
# 5. Confusion Matrices for All Three Models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# AdaBoost
sns.heatmap(ada_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=le.classes_, yticklabels=le.classes_)
axes[0].set_title(f'AdaBoost\nAccuracy: {ada_accuracy:.2%}', fontsize=14, fontweight='bold')
axes[0].set_ylabel('True Label', fontsize=12)
axes[0].set_xlabel('Predicted Label', fontsize=12)

# Gradient Boosting
sns.heatmap(gb_cm, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=le.classes_, yticklabels=le.classes_)
axes[1].set_title(f'Gradient Boosting\nAccuracy: {gb_accuracy:.2%}', fontsize=14, fontweight='bold')
axes[1].set_ylabel('True Label', fontsize=12)
axes[1].set_xlabel('Predicted Label', fontsize=12)

# XGBoost
sns.heatmap(xgb_cm, annot=True, fmt='d', cmap='Purples', ax=axes[2],
            xticklabels=le.classes_, yticklabels=le.classes_)
axes[2].set_title(f'XGBoost\nAccuracy: {xgb_accuracy:.2%}', fontsize=14, fontweight='bold')
axes[2].set_ylabel('True Label', fontsize=12)
axes[2].set_xlabel('Predicted Label', fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(img_dir, 'boosting_confusion_matrices.png'), dpi=300, bbox_inches='tight')
plt.close()

print("✓ Saved boosting_confusion_matrices.png")


In [ ]:
# 6. Feature Importance Comparison
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# AdaBoost feature importance
ada_importance = pd.DataFrame({
    'feature': numeric_features,
    'importance': ada_best_model.feature_importances_
}).sort_values('importance', ascending=True)

axes[0].barh(ada_importance['feature'], ada_importance['importance'], color='skyblue')
axes[0].set_xlabel('Importance', fontsize=12)
axes[0].set_title('AdaBoost\nFeature Importance', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# Gradient Boosting feature importance
gb_importance = pd.DataFrame({
    'feature': numeric_features,
    'importance': gb_best_model.feature_importances_
}).sort_values('importance', ascending=True)

axes[1].barh(gb_importance['feature'], gb_importance['importance'], color='lightgreen')
axes[1].set_xlabel('Importance', fontsize=12)
axes[1].set_title('Gradient Boosting\nFeature Importance', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

# XGBoost feature importance
xgb_importance = pd.DataFrame({
    'feature': numeric_features,
    'importance': xgb_best_model.feature_importances_
}).sort_values('importance', ascending=True)

axes[2].barh(xgb_importance['feature'], xgb_importance['importance'], color='plum')
axes[2].set_xlabel('Importance', fontsize=12)
axes[2].set_title('XGBoost\nFeature Importance', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(os.path.join(img_dir, 'boosting_feature_importance.png'), dpi=300, bbox_inches='tight')
plt.close()

print("✓ Saved boosting_feature_importance.png")


In [ ]:
# 7. Accuracy Comparison Across Hyperparameters
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# AdaBoost: n_estimators vs accuracy for each learning rate
ada_df = pd.DataFrame(adaboost_results)
for lr in learning_rates:
    subset = ada_df[ada_df['learning_rate'] == lr]
    axes[0].plot(subset['n_estimators'], subset['accuracy'], marker='o', label=f'LR={lr}', linewidth=2)
axes[0].set_xlabel('Number of Estimators', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('AdaBoost\nHyperparameter Tuning', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gradient Boosting: n_estimators vs accuracy for different depths at lr=0.1
gb_df = pd.DataFrame(gb_results)
gb_subset = gb_df[gb_df['learning_rate'] == 0.1]
for depth in max_depths:
    subset = gb_subset[gb_subset['max_depth'] == depth]
    axes[1].plot(subset['n_estimators'], subset['accuracy'], marker='s', label=f'Depth={depth}', linewidth=2)
axes[1].set_xlabel('Number of Estimators', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Gradient Boosting\nHyperparameter Tuning (LR=0.1)', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# XGBoost: learning rate vs accuracy for different depths at n_est=100
xgb_df = pd.DataFrame(xgb_results)
xgb_subset = xgb_df[xgb_df['n_estimators'] == 100]
for depth in [3, 5, 7]:
    subset = xgb_subset[xgb_subset['max_depth'] == depth]
    axes[2].plot(subset['learning_rate'], subset['accuracy'], marker='^', label=f'Depth={depth}', linewidth=2)
axes[2].set_xlabel('Learning Rate', fontsize=12)
axes[2].set_ylabel('Accuracy', fontsize=12)
axes[2].set_title('XGBoost\nHyperparameter Tuning (n_est=100)', fontsize=14, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(img_dir, 'boosting_accuracy_comparison.png'), dpi=300, bbox_inches='tight')
plt.close()

print("✓ Saved boosting_accuracy_comparison.png")


In [ ]:
# 8. Overall Performance Comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart of accuracies
models = ['AdaBoost', 'Gradient\nBoosting', 'XGBoost']
accuracies = [ada_accuracy, gb_accuracy, xgb_accuracy]
colors_bars = ['#3498db', '#2ecc71', '#9b59b6']

bars = axes[0].bar(models, accuracies, color=colors_bars, alpha=0.8, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Accuracy', fontsize=14)
axes[0].set_title('Model Accuracy Comparison', fontsize=16, fontweight='bold')
axes[0].set_ylim([0, 1.0])
axes[0].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{acc:.2%}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Get classification reports for comparison
ada_metrics = precision_recall_fscore_support(y_test, ada_predictions, average='weighted')
gb_metrics = precision_recall_fscore_support(y_test, gb_predictions, average='weighted')
xgb_metrics = precision_recall_fscore_support(y_test_encoded, xgb_predictions, average='weighted')

metrics_df = pd.DataFrame({
    'Model': models,
    'Precision': [ada_metrics[0], gb_metrics[0], xgb_metrics[0]],
    'Recall': [ada_metrics[1], gb_metrics[1], xgb_metrics[1]],
    'F1-Score': [ada_metrics[2], gb_metrics[2], xgb_metrics[2]]
})

x = np.arange(len(models))
width = 0.25

axes[1].bar(x - width, metrics_df['Precision'], width, label='Precision', color='#e74c3c', alpha=0.8)
axes[1].bar(x, metrics_df['Recall'], width, label='Recall', color='#f39c12', alpha=0.8)
axes[1].bar(x + width, metrics_df['F1-Score'], width, label='F1-Score', color='#16a085', alpha=0.8)

axes[1].set_ylabel('Score', fontsize=14)
axes[1].set_title('Precision, Recall, and F1-Score Comparison', fontsize=16, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models)
axes[1].legend()
axes[1].set_ylim([0, 1.0])
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(img_dir, 'boosting_performance_comparison.png'), dpi=300, bbox_inches='tight')
plt.close()

print("✓ Saved boosting_performance_comparison.png")


## Conclusions

### Key Findings:

1. **Best Performing Model**: Based on accuracy and other metrics, the best model is indicated above

2. **Model Characteristics**:
   - **AdaBoost**: Simple and effective, works by reweighting samples
   - **Gradient Boosting**: More sophisticated, fits residuals iteratively
   - **XGBoost**: Most advanced with regularization and optimization

3. **Feature Importance**: Different algorithms prioritize features differently, showing varied perspectives on what matters for round outcomes

4. **Hyperparameter Sensitivity**: All models showed sensitivity to learning rate and number of estimators, with XGBoost being most stable

5. **Practical Implications**: These results can help understand CS2 gameplay dynamics and inform strategic decisions


In [ ]:
print("\n" + "="*80)
print("BOOSTING ANALYSIS COMPLETE")
print("="*80)
print(f"\nTotal rounds analyzed: {len(features_df_clean)}")
print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
print(f"\nBest Model: {best_model_name}")
print(f"Best Accuracy: {best_accuracy:.4f}")
print(f"\nAll visualizations saved to: {img_dir}")
print("="*80)
